In [2]:
import requests
import tensorflow as tf
import base64
import nltk
import re
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from pprint import PrettyPrinter
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/acqmallatief/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/acqmallatief/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [33]:
pp = PrettyPrinter()
pp.pprint(requests.get("http://localhost:8080/v1/models/emotion-classification-model").json())

feel realli helpless heavi heart


In [2]:
stemmer = PorterStemmer()
stop_words = stopwords.words('english')

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub('\w*\d\w*', '', text)
    tokens = word_tokenize(text)
    cleaned_tokens = [stemmer.stem(token) for token in tokens if token not in stop_words]
    cleaned_text = ' '.join(cleaned_tokens)
    return cleaned_text

def _bytes_feature(value):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def serialize_text(text):
    example = tf.train.Example(features=tf.train.Features(feature={
        'text': _bytes_feature(text)
        }))
    serialized_example = example.SerializeToString()
    return serialized_example

In [8]:
end_point = 'http://localhost:8080/v1/models/emotion-classification-model:predict'

def predict(input):
    text = clean_text(input)
    text = text.encode('utf-8')
    example = serialize_text(input)
    json_data = {
        "signature_name":"serving_default",
        "instances":[
            {
                "examples":{"b64": base64.b64encode(example).decode('utf-8')}
            }
        ]
    }
    response = requests.post(end_point, json=json_data)
    prediction = tf.argmax(response.json()["predictions"][0]).numpy()
    map_labels = {0: "sadness",
                  1: "joy",
                  2: "love",
                  3: "anger",
                  4: "fear",
                  5: "surprise"}
    return map_labels[prediction]

In [12]:
text = input("Masukkan text:")

print(f"{text}\n classifisi:{predict(text)}")

classifisi:joy


In [6]:
END_POINT = "output/serving_model/1709760128"
MODEL = tf.keras.models.load_model(END_POINT)

test = MODEL.predict("test i love")
prediction = tf.argmax(test[0]).numpy()
map_labels = {0: "sadness",
                  1: "joy",
                  2: "love",
                  3: "anger",
                  4: "fear",
                  5: "surprise"}
test

2024-03-08 08:04:55.209768: W tensorflow/core/common_runtime/graph_constructor.cc:840] Node 'cond/while' has 13 outputs but the _output_shapes attribute specifies shapes for 46 outputs. Output shapes may be inaccurate.
2024-03-08 08:04:55.298596: W tensorflow/core/common_runtime/graph_constructor.cc:840] Node 'cond/while' has 13 outputs but the _output_shapes attribute specifies shapes for 46 outputs. Output shapes may be inaccurate.
2024-03-08 08:04:55.544834: W tensorflow/core/common_runtime/graph_constructor.cc:840] Node 'cond/while' has 13 outputs but the _output_shapes attribute specifies shapes for 46 outputs. Output shapes may be inaccurate.
2024-03-08 08:04:55.638459: W tensorflow/core/common_runtime/graph_constructor.cc:840] Node 'cond' has 5 outputs but the _output_shapes attribute specifies shapes for 46 outputs. Output shapes may be inaccurate.
2024-03-08 08:04:55.695272: W tensorflow/core/common_runtime/graph_constructor.cc:840] Node 'cond/while' has 13 outputs but the _ou

IndexError: tuple index out of range